# Solve an existing problem with a new algorithm

**Scenario:** the problems you care about already exist, and you want to try a new algorithm
or a variant of an existing one.

The first question is always: **what does the algorithm need from a problem?**

- **The needs are covered by an existing requirements interface.** Depend on that
  interface, and every problem implementing it works immediately (path A).
- **The algorithm needs something new.** Define a new requirements interface and implement
  it in the problems you want to support (path B).

We walk through one example of each.

In [1]:
from abc import ABC, abstractmethod
from collections.abc import Callable

import networkx as nx
import numpy as np
from qrisp import QuantumArray, QuantumVariable

from qrisp_optimization import QAOA, Bruteforce, CLCostRequirements, QAOARequirements
from qrisp_optimization.metrics import exp_val
from qrisp_optimization.problems import MaxCutProblem, QUBOProblem
from qrisp_optimization.utils import QuantumBinary

We use the same weighted MaxCut instance for both paths, plus its exact solutions as a
reference.

In [2]:
G = nx.Graph()
G.add_weighted_edges_from(
    [(0, 1, 1.0), (0, 2, 2.0), (1, 2, 1.0), (1, 3, 3.0), (2, 4, 1.0), (3, 4, 2.0), (4, 5, 1.0)]
)
problem = MaxCutProblem(G)

optimal = Bruteforce(problem).solve()
optimal_cost = problem.cl_cost_function(optimal[0])
optimal_set = {tuple(x) for x in optimal}
print("optimal cost:", optimal_cost)

optimal cost: -10.0


## Path A: reuse an existing interface

As an example we implement **INTERP-QAOA**
([Zhou et al., 2020](https://doi.org/10.1103/PhysRevX.10.021067)), a strategy to optimize deep
QAOA circuits. Instead of optimizing depth `p` from scratch, it optimizes depth 1, then
linearly *interpolates* the optimal angles to a starting point for depth 2, optimizes again,
and so on.

This only changes *how the angles are chosen*. The circuit is still state preparation
followed by alternating cost and mixer layers, so
[`QAOARequirements`][qrisp_optimization.QAOARequirements] covers everything the algorithm
needs.

The algorithm takes the problem typed as the **interface**, never as a concrete class.
Here it even reuses [`QAOA`][qrisp_optimization.QAOA] for the circuit and the single-depth
optimization.

In [3]:
class InterpQAOA[T]:
    # QAOA whose angles are grown layer by layer with the INTERP heuristic.

    def __init__(
        self, problem: QAOARequirements[T], qarg: Callable[[], QuantumVariable | QuantumArray]
    ) -> None:
        self.problem = problem
        self.qaoa = QAOA(problem, qarg)

    @staticmethod
    def interp(angles: np.ndarray) -> np.ndarray:
        # extend p optimal angles to a p+1 starting point (Zhou et al., eq. B1)
        p = len(angles)
        ext = np.concatenate([[0.0], angles, [0.0]])
        i = np.arange(1, p + 2)
        return (i - 1) / p * ext[i - 1] + (p - i + 1) / p * ext[i]

    def ansatz(self, theta: np.ndarray) -> QuantumVariable | QuantumArray:
        return self.qaoa.ansatz(theta)

    def optimize(
        self, max_depth: int, theta1: np.ndarray | None = None, max_iter: int = 100
    ) -> list[np.ndarray]:
        # returns the optimized angles for every depth 1..max_depth
        theta = np.array([0.5, 0.5]) if theta1 is None else theta1
        history = []
        for p in range(1, max_depth + 1):
            theta = self.qaoa.optimize(theta, max_iter=max_iter)
            history.append(theta)
            betas, gammas = theta[:p], theta[p:]
            theta = np.concatenate([self.interp(betas), self.interp(gammas)])
        return history

Use it like any other algorithm:

In [4]:
interp_qaoa = InterpQAOA(
    problem=problem,
    qarg=lambda: QuantumArray(qtype=QuantumBinary(), shape=problem.n),
)
history = interp_qaoa.optimize(max_depth=4)

for theta in history:
    qarg = interp_qaoa.ansatz(theta)
    meas = qarg.get_measurement()
    p_opt = sum(prob for x, prob in meas.items() if tuple(x) in optimal_set)
    print(f"p={len(theta) // 2}:  <C> = {exp_val(qarg, problem):7.3f}   P(optimal) = {p_opt:.3f}")

p=1:  <C> =  -8.013   P(optimal) = 0.170


p=2:  <C> =  -8.846   P(optimal) = 0.279


p=3:  <C> =  -9.153   P(optimal) = 0.395


p=4:  <C> =  -9.408   P(optimal) = 0.539


Since `InterpQAOA` only depends on `QAOARequirements`, it works on every problem that
implements that interface, without any further changes:

In [5]:
Q = np.array(
    [
        [1.0, -3.0, -2.0, 2.0],
        [0.0, 2.5, -2.0, 0.0],
        [0.0, 0.0, 3.0, -2.5],
        [0.0, 0.0, 0.0, 1.0],
    ]
)
qubo = QUBOProblem(Q)
qubo_algo = InterpQAOA(qubo, qarg=lambda: QuantumArray(qtype=QuantumBinary(), shape=qubo.n))
theta = qubo_algo.optimize(max_depth=3)[-1]

meas = qubo_algo.ansatz(theta).get_measurement()
x, prob = max(meas.items(), key=lambda kv: kv[1])
print("bruteforce: ", [x.tolist() for x in Bruteforce(qubo).solve()])
print(f"most likely: {list(x)} with p={prob:.3f}")

bruteforce:  [[1, 1, 1, 0]]
most likely: [1, 1, 1, 0] with p=0.493


## Path B: define a new requirements interface

Now suppose the algorithm needs something no existing interface provides. As a deliberately
simple example we use classical **simulated annealing**, a common baseline for quantum
optimizers. It walks through the search space by repeatedly proposing a *neighboring*
solution, so it needs a problem to

- evaluate a solution: `cl_cost_function`, already in
  [`CLCostRequirements`][qrisp_optimization.CLCostRequirements],
- draw a random starting solution: **new**,
- propose a neighbor of a solution: **new**.

> In this case you have to do all the work anyway, so you might as well stick to the
> architecture.

### 1. Define the interface

Name it after the algorithm and extend `CLCostRequirements`, so metrics and other generic
tooling keep working with it.

In [6]:
class AnnealingRequirements[T](CLCostRequirements[T], ABC):
    # Abstract interface for problems solvable by SimulatedAnnealing.

    @abstractmethod
    def random_solution(self, rng: np.random.Generator) -> T:
        # draw a random candidate solution
        ...

    @abstractmethod
    def neighbor(self, x: T, rng: np.random.Generator) -> T:
        # return a random solution close to x (without modifying x)
        ...

### 2. Implement the algorithm against the interface

In [7]:
class SimulatedAnnealing[T]:
    # Classical simulated annealing with a geometric temperature schedule.

    def __init__(
        self,
        problem: AnnealingRequirements[T],
        steps: int = 2000,
        t_start: float = 5.0,
        t_end: float = 0.01,
        seed: int | None = None,
    ) -> None:
        self.problem = problem
        self.steps = steps
        self.t_start = t_start
        self.t_end = t_end
        self.rng = np.random.default_rng(seed)

    def solve(self) -> tuple[T, float]:
        x = self.problem.random_solution(self.rng)
        cost = self.problem.cl_cost_function(x)
        best, best_cost = x, cost

        for t in np.geomspace(self.t_start, self.t_end, self.steps):
            y = self.problem.neighbor(x, self.rng)
            y_cost = self.problem.cl_cost_function(y)
            if y_cost <= cost or self.rng.random() < np.exp((cost - y_cost) / t):
                x, cost = y, y_cost
                if cost < best_cost:
                    best, best_cost = x, cost

        return best, best_cost

### 3. Make the existing problems satisfy the interface

In the package, you would add `AnnealingRequirements` to the base classes of
[`MaxCutProblem`][qrisp_optimization.problems.MaxCutProblem] and implement the two methods
there. In this notebook we subclass instead. For binary problems, a neighbor is a single
bit flip.

In [8]:
class AnnealableMaxCut(MaxCutProblem, AnnealingRequirements):
    def random_solution(self, rng: np.random.Generator) -> np.ndarray:
        return rng.integers(0, 2, size=self.n)

    def neighbor(self, x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
        y = x.copy()
        y[rng.integers(self.n)] ^= 1
        return y


problem = AnnealableMaxCut(G)

### 4. Use it

The problem still satisfies all its previous interfaces, so the same instance works with
the new algorithm *and* with all existing ones:

In [9]:
x, cost = SimulatedAnnealing(problem, seed=1).solve()
print(f"simulated annealing: {x}  cost={cost}  optimal={tuple(x) in optimal_set}")
print(f"bruteforce:          cost={problem.cl_cost_function(Bruteforce(problem).solve()[0])}")

simulated annealing: [0 0 1 1 0 1]  cost=-10.0  optimal=True
bruteforce:          cost=-10.0


## Adding the algorithm to the package

1. Put the algorithm in `qrisp_optimization/<algorithm>.py` and, for path B, the interface in
   `qrisp_optimization/<algorithm>_requirements.py`, following the existing
   `qaoa.py` / `qaoa_requirements.py` pair. Use NumPy-style docstrings.
2. Export both in `qrisp_optimization/__init__.py`.
3. For path B, add the interface to every problem class that should support the algorithm.
4. Add `tests/test_<algorithm>.py`. Comparing against `Bruteforce` on small instances is a
   good default, see `tests/test_qaoa.py`.
5. Run `scripts/check.sh` (ruff, mypy, pytest).